# 20.8 訓練跑完後，怎麼決定可以交付哪個版本？


照片描述變好了，聊天卻退步，要不要採用這份修正？這要由事先約定好的用途與門檻決定，不能事後只挑進步的那幾欄。本課的比較讓原底座與 LoRA 版本使用相同的圖片處理與生成設定，Whisper 的版本和它實際產生的轉寫文字也固定，差別只在有沒有開啟修正。候選有兩個：同一次訓練在第 1,039 次與第 2,077 次更新後，各保存一份修正，下面簡稱 1,039 步與 2,077 步。

修正版要取代原底座，必須同時通過三項條件：

1. 回答完整結束：全部 132 份驗證回答都以 EOS 結束。生成的上限是 384 個新 token，用滿上限仍沒有 EOS，就算截斷。
2. 每種用途達到保留門檻：答對題數不能比原底座少太多。照片短描述、指定讀序與兩條語音問句聊天，一題都不能少；其餘用途最多少 1 題。下表最右欄是由此算出、修正版至少要答對的題數。
3. 綜合分數嚴格高於原底座，算法在表後。

| 同一份驗證題的用途 | 原底座 | 1,039 步 | 2,077 步 | 修正版至少答對 |
| --- | ---: | ---: | ---: | ---: |
| 照片短描述 | 16／28 | 23／28 | 24／28 | 16 |
| 照片可見事實問答 | 31／56 | 53／56 | 51／56 | 30 |
| 有沒有清楚可見的中文字 | 18／18 | 18／18 | 18／18 | 17 |
| 單一區域逐字抄寫 | 6／10 | 5／10 | 5／10 | 5 |
| 指定讀序與分行 | 2／3 | 2／3 | 2／3 | 2 |
| 一般文字聊天 | 3／9 | 2／9 | 5／9 | 2 |
| 真人問句：直接給正確轉寫文字後聊天 | 4／4 | 2／4 | 1／4 | 4 |
| 同組問句：給 ASR 轉寫文字後聊天 | 3／4 | 2／4 | 2／4 | 3 |

照片兩項大幅改善，但兩個候選都在最後兩列跌破門檻。直接給正確轉寫文字的那一列完全不經過 ASR，1,039 步仍從 4／4 降到 2／4，可見退步不全是聽錯造成的；走真正 ASR 的那一列也下降。兩個候選也各有 3 份回答用滿 384 個新 token 仍沒有結束，第一項條件同樣沒有通過。

綜合分數把八種用途分成五組能力，各組先算正確率：照片組取前兩列兩個正確率的平均，聊天組取最後三列三個正確率的平均，中間三列各自成一組；五組再平均，每組各占五分之一。這樣題數多的照片題，不會因為題多就蓋過其他用途。原底座的綜合分數是 (1/5) × [(16/28 + 31/56)/2 + 18/18 + 6/10 + 2/3 + (3/9 + 4/4 + 3/4)/3] ＝ 53,277/75,600，約 70.47%。75,600 是讓每一項都化成整數分子的共同分母，比較時就不必四捨五入。下面照抄選版程式 [scripts/score_natural_v4_validation.py](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/scripts/score_natural_v4_validation.py) 的 `WEIGHTS`（每列答對一題得幾分）與 `DENOMINATOR`（滿分），用上表的答對題數重算三個版本，並檢查第 2、3 項條件：


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# 每題分數與滿分照抄自 scripts/score_natural_v4_validation.py 的 WEIGHTS、DENOMINATOR
weights = [270, 135, 840, 1512, 5040, 560, 1260, 1260]
denominator = 75600
allowed_drop = [0, 1, 1, 1, 0, 1, 0, 0]  # 保留門檻：各列最多可比原底座少答對幾題
base = [16, 31, 18, 6, 2, 3, 4, 3]
candidates = {"1,039步": [23, 53, 18, 5, 2, 2, 2, 2], "2,077步": [24, 51, 18, 5, 2, 5, 1, 2]}
base_score = sum(c * w for c, w in zip(base, weights))
minimum = [b - d for b, d in zip(base, allowed_drop)]
print("原底座", "綜合分數", f"{base_score}/{denominator}", "百分比", round(100 * base_score / denominator, 2))
print("修正版至少答對", minimum)
for name, counts in candidates.items():
    score = sum(c * w for c, w in zip(counts, weights))
    passed = [c >= m for c, m in zip(counts, minimum)]
    print(name, "綜合分數", f"{score}/{denominator}", "百分比", round(100 * score / denominator, 2))
    print(name, "高於原底座", score > base_score, "各列達到門檻", passed)

每份清單都照上表由上到下排列；`weights` 的第一項 270＝75,600×1/5×1/2×1/28，也就是照片組占的五分之一先由兩列平分，再分給該列的 28 題。「修正版至少答對」由 `allowed_drop` 算出，和上表最右欄相同；`score > base_score` 就是「嚴格高於原底座」。程式算出 1,039 步是 52,285/75,600，約 69.16%；2,077 步是 52,705/75,600，約 69.72%；兩者的「各列達到門檻」都只有最後兩項是 False。照片組的進步被聊天組與單區讀字的下降抵掉，兩個候選的綜合分數都比原底座低。

練習把 `weights` 的八項都改成 1、`denominator` 改成 132（全部驗證題數），讓每題同分。先預測：照片兩列占 132 題裡的 84 題，它們的進步會蓋過其他列的退步，兩個候選變成 107/132 與 108/132，都高於原底座的 83/132；「各列達到門檻」不變，所以仍然不能選。

三項條件都沒有通過，所以本版[選定原底座](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/v4/selection.json)（`selected_variant` 欄是 `base`），搭配相同的 Whisper-large-v3-turbo，不開 LoRA。較晚保存的 2,077 步，也沒有因為訓練得比較久就自動取得交付資格。這些分母很小，部分題目共用同一張照片；開放式描述與聊天的對錯依約定判準逐題判讀，換一位判讀者可能有出入。所以結果只支持這組比較。

練習用上表核對 2,077 步：它的一般文字聊天 5／9 比原底座多答對 2 題，為什麼仍不能選？答案思路：門檻要每一列各自通過，它的正確轉寫文字聊天 1／4 低於 4、ASR 轉寫文字聊天 2／4 低於 3，其他用途的進步不能抵銷。

訓練代價下降，只說明模型對示範答案的預測變了；完整生成的回答仍可能重複，或多添畫面沒有的細節。正確的順序是：先在驗證資料上決定配置，再凍結權重與生成設定，最後才用測試資料描述它的能力（[20.13](https://birdhackor.github.io/tiny-perceptron-vlm/20.13.html)），測試結果不再拿來挑模型。原底座自己也會錯，這次驗證中也有 6 份回答沒有正常結束；保留它，只表示兩個候選沒有提供足夠的採用理由。

<details>
<summary>補充：候選訓練、判準與原始驗證</summary>

候選從固定底座開始更新 LoRA，在第 1,039 與 2,077 次更新後保存。訓練命令與更新條件見[訓練指引第4節](https://birdhackor.github.io/tiny-perceptron-vlm/natural-v4-training.html#4-保存配方開始自己的新訓練)，驗證命令與完整門檻見[第6節](https://birdhackor.github.io/tiny-perceptron-vlm/natural-v4-training.html#6-先驗證用途再選定自己的版本)。各分項與完整生成見[驗證得分](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/evidence/v4-runtime/validation-37219466611/blind-review/scored/scores.json)：`variants` 底下每個版本的 `correct_counts` 是上表的答對題數，`primary_numerator` 是綜合分數的分子（分母 75,600），`gates` 列出每道門檻的最低題數與是否通過。本節沒有重新訓練或重新選版。

</details>
